# Text Generation using LSTMs

In [1]:
import os
import urllib.request

import numpy

from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM
from keras.utils import np_utils

Using TensorFlow backend.


## Download the data

The best place to access books that are no longer under Copyright is [Project Gutenberg](https://www.gutenberg.org/). Today we recommend using [Alice’s Adventures in Wonderland by Lewis Carroll](https://www.gutenberg.org/files/11/11-0.txt) for consistency. Of course you can experiment with other books as well.

In [0]:
data_url = 'https://www.gutenberg.org/files/219/219-0.txt'
fname = 'heart_of_darkness.txt'

if fname not in os.listdir():
    urllib.request.urlretrieve(data_url, fname)

## Load data and create character to integer mappings

- Open the text file, read the data then convert it to lowercase letters.
- Map each character to a respective number. Keep 2 dictionaries in order to have more easily access to the mappings both ways around.

In [0]:
# Load data
text_data = (open(fname, 'r', encoding='utf-8').read())
text_data = text_data.lower()

# Characters to integers
unique_ch = sorted(list(set(text_data)))

char_to_int = {}
int_to_char = {}

for i, c in enumerate(unique_ch):
    char_to_int.update({c: i})
    int_to_char.update({i: c})

## Prepare the data
- We are "thinking" in sequences of 100 characters: 99 characters in the input and 1 in the output.  
E.g. for the sequence *\['h', 'e', 'l', 'l'\]* as input, we will have *\['o'\]* as the expected output.
- Reshape X such that it has the shape expected by a LSTM: \[samples, time steps, features\].
  - samples: number of data points (len(X));
  - time steps: number of time-dependent steps that are in a single data point (100);
  - features: number of variables for the true value in Y (1).
- Scale the values in X to be in \[0, 1\].
- One-hot encode the true values in Y_modified.

In [0]:
# Initialize the input and output with empty lists
X = []; Y = []

for i in range(0, len(text_data) - 100, 1):
    # Consider sequences of 99 characters starting from i
    seq = text_data[i:i + 100]
    # The 50th character is the label
    label = text_data[i + 100]

    # Append to the input the list of ints corresponding to the characters in the current sequence
    X.append([char_to_int[ch] for ch in seq])
    # Append to the output the int corresponding to the label (as list)
    Y.append(char_to_int[label])

# Re-shape the inputs
X_new = numpy.reshape(X, (len(X), 100, 1))
# Scale the inputs
X_new = X_new / float(len(unique_ch))
# One-hot encode labels
Y_new = np_utils.to_categorical(Y)

## Define the LSTM model

- Instantiate the model: a linear stack of layers.
- First layer: LSTM with 256 memory units, input shape from X_new (1st and 2nd). Make sure that this layer returns sequences, such that the next LSTM layer receives sequences and not just random data.
- Second layer: dropout 20% of the neurons of the previous layer in order to avoid overfitting.

****** 
Optional:
- Third layer: LSTM(256).
- Fourth layer: dropout 20% of the neurons.
******
- Last layer: fully connected with a 'softmax' activation function, and as many neurons as the number of unique characters (the output is one-hot encoded).


Compile the model: categorical_crossentropy, adam.

In [5]:
# Instantiate the model
model = Sequential()

# Add LSTM layer
model.add(LSTM(256, input_shape=(X_new.shape[1], X_new.shape[2])))

# Add dropout
# model.add(Dropout(0.2))

# Add another LSTM layer
#model.add(LSTM(256))

# Add dropout
#model.add(Dropout(0.2))

# Add a Dense layer
model.add(Dense(Y_new.shape[1], activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam')

## Train the model and generate characters

Fit the model for over 100 epochs as the batch size is 30 (ideally). In this case, given the time constraints, we are going to use 5 epochs and a batch size of 128. 

Fix a random seed and start generating characters.  The prediction from the model gives out the character encoding of the predicted character, it is then decoded back to the character value and appended to the pattern.  

After enough training time it is going to look like something.

In [6]:
from keras.callbacks import ModelCheckpoint

filepath="weights-improvement-{epoch:02d}-{loss:.4f}.hdf5"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=2, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

# fit the model
history=model.fit(X_new, Y_new, epochs=5, batch_size=128, callbacks=callbacks_list)

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where



Epoch 1/5





561795/561795 [==============================] - 257s 458us/step - loss: 2.8465

Epoch 00001: loss improved from inf to 2.84652, saving model to weights-improvement-01-2.8465.hdf5
Epoch 2/5
561795/561795 [==============================] - 253s 450us/step - loss: 2.6919

Epoch 00002: loss improved from 2.84652 to 2.69188, saving model to weights-improvement-02-2.6919.hdf5
Epoch 3/5
561795/561795 [==============================] - 259s 461us/step - loss: 2.6167

Epoch 00003: loss improved from 2.69188 to 2.61674, saving model to weights-improvement-03-2.6167.hdf5
Epoch 4/5
561795/561795 [==============================] - 255s 455us/step - loss: 2.5577

Epoch 00004: loss improved from 2.61674 to 2.55774, saving model to weights-improvement-04-2.5577.hdf5
Epoch 5/5
561795/561795 [==============================] - 253s 450us/step - loss: 2.5081

Epoch 00005: loss improved from 2.5

In [7]:
# pick a random seed
start_index = numpy.random.randint(0, len(X)-1)
new_string = X[start_index]

a = "\"" + ''.join([int_to_char[value] for value in new_string]) + "\""

print("Seed:" + str(a) + "\n")

# generate characters
for i in range(100):
    x = numpy.reshape(new_string, (1, len(new_string), 1))
    x = x / float(len(unique_ch))

    #predict
    pred_index = numpy.argmax(model.predict(x, verbose=0))
    char_out = int_to_char[pred_index]
    seq_in = [int_to_char[value] for value in new_string]
    a += char_out

    new_string.append(pred_index)
    new_string = new_string[1:len(new_string)]

print(a)

Seed:", together
with a hurried scrawl, tellin"

", together
with a hurried scrawl, tellin" the sooe to the sooe to the sooe to the sooe to t


# Bonus: Words as features

Code here:

https://machinelearningmastery.com/how-to-develop-a-word-level-neural-language-model-in-keras/ 